In [36]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# Plot settings
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

In [37]:
# Load the cleaned data
df = pd.read_csv('Telco_Cleaned_Model_Ready.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes.value_counts())

print("\nTarget Distribution (Churn):")
print(df['Churn'].value_counts(normalize=True))

Dataset Shape: (7043, 32)

First 5 rows:


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,Service_Intensity,Contract_Rank,Value_at_Risk,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,29.85,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,2,1,1936.30,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True
2,0,2,53.85,108.15,1,2,0,107.70,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,3,1,1903.50,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,False,False
4,0,2,70.70,151.65,1,0,0,141.40,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False



Data Types:
bool       24
int64       5
float64     3
Name: count, dtype: int64

Target Distribution (Churn):
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [38]:
# Convert boolean columns True/False to integers 0/1
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
print(f"Converting {len(bool_cols)} boolean columns to integers...")

df[bool_cols] = df[bool_cols].astype(int)

print("Data types after conversion:")
print(df.dtypes.value_counts())

Converting 24 boolean columns to integers...
Data types after conversion:
int64      29
float64     3
Name: count, dtype: int64


In [39]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Train-test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Churn rate in train: {y_train.mean():.4f}")
print(f"Churn rate in test: {y_test.mean():.4f}")

Training set shape: (5634, 31)
Test set shape: (1409, 31)
Churn rate in train: 0.2654
Churn rate in test: 0.2654


In [40]:
# Baseline Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("=== BASELINE RANDOM FOREST PERFORMANCE ===")
print(classification_report(y_test, rf_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, rf_prob):.4f}")
print(f"F1-Score (Churn class): {f1_score(y_test, rf_pred, pos_label=1):.4f}")

=== BASELINE RANDOM FOREST PERFORMANCE ===
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

AUC-ROC: 0.8265
F1-Score (Churn class): 0.5554


In [41]:
# Baseline XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("=== BASELINE XGBOOST PERFORMANCE ===")
print(classification_report(y_test, xgb_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, xgb_prob):.4f}")
print(f"F1-Score (Churn class): {f1_score(y_test, xgb_pred, pos_label=1):.4f}")

=== BASELINE XGBOOST PERFORMANCE ===
              precision    recall  f1-score   support

           0       0.87      0.80      0.83      1035
           1       0.55      0.67      0.60       374

    accuracy                           0.77      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.77      0.77      1409

AUC-ROC: 0.8227
F1-Score (Churn class): 0.6043


In [42]:
# Compare baseline models
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost (Baseline)'],
    'Accuracy': [rf_model.score(X_test, y_test), xgb_model.score(X_test, y_test)],
    'AUC-ROC': [roc_auc_score(y_test, rf_prob), roc_auc_score(y_test, xgb_prob)],
    'F1-Score (Churn Class)': [
        f1_score(y_test, rf_pred, pos_label=1),
        f1_score(y_test, xgb_pred, pos_label=1)
    ]
})

display(comparison.round(4))

,Model,Accuracy,AUC-ROC,F1-Score (Churn Class)
0,Random Forest,0.7921,0.8265,0.5554
1,XGBoost (Baseline),0.7658,0.8227,0.6043


In [43]:
# Hyperparameter tuning for XGBoost
print("Starting hyperparameter tuning for XGBoost...")

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_tuned = xgb.XGBClassifier(
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc'
)

grid_search = GridSearchCV(
    estimator=xgb_tuned,
    param_grid=param_grid,
    scoring='f1',        
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best F1 score:", grid_search.best_score_)

# Train final tuned model
best_xgb = grid_search.best_estimator_
best_xgb_pred = best_xgb.predict(X_test)
best_xgb_prob = best_xgb.predict_proba(X_test)[:, 1]

print("\n=== TUNED XGBOOST PERFORMANCE ===")
print(classification_report(y_test, best_xgb_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, best_xgb_prob):.4f}")
print(f"F1-Score (Churn class): {f1_score(y_test, best_xgb_pred, pos_label=1):.4f}")

Starting hyperparameter tuning for XGBoost...
Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}
Best F1 score: 0.6324241872645403

=== TUNED XGBOOST PERFORMANCE ===
              precision    recall  f1-score   support

           0       0.91      0.74      0.81      1035
           1       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.77      1409

AUC-ROC: 0.8442
F1-Score (Churn class): 0.6290


In [44]:
# Threshold tuning - find best probability cutoff for F1 on churn class
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, best_xgb_prob)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx]

print(f"Best threshold for F1-score: {best_threshold:.4f}")
print(f"Best F1-score at that threshold: {f1_scores[best_threshold_idx]:.4f}")

# Apply the best threshold
tuned_pred = (best_xgb_prob >= best_threshold).astype(int)

print("\n=== PERFORMANCE WITH OPTIMIZED THRESHOLD ===")
print(classification_report(y_test, tuned_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, best_xgb_prob):.4f}")

Best threshold for F1-score: 0.5155
Best F1-score at that threshold: 0.6336

=== PERFORMANCE WITH OPTIMIZED THRESHOLD ===
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      1035
           1       0.53      0.79      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409

AUC-ROC: 0.8442


In [45]:
# Final Model Comparison
final_comparison = pd.DataFrame({
    'Model': ['Baseline XGBoost', 'Tuned XGBoost (0.5 threshold)', 'Tuned + Optimized Threshold'],
    'AUC-ROC': [
        roc_auc_score(y_test, xgb_prob),
        roc_auc_score(y_test, best_xgb_prob),
        roc_auc_score(y_test, best_xgb_prob)
    ],
    'F1-Score (Churn Class)': [
        f1_score(y_test, xgb_pred, pos_label=1),
        f1_score(y_test, best_xgb_pred, pos_label=1),
        f1_score(y_test, tuned_pred, pos_label=1)
    ]
})

display(final_comparison.round(4))

,Model,AUC-ROC,F1-Score (Churn Class)
0,Baseline XGBoost,0.8227,0.6043
1,Tuned XGBoost (0.5 threshold),0.8442,0.6290
2,Tuned + Optimized Threshold,0.8442,0.6336


In [46]:
# Final High-Risk Customer List using Tuned XGBoost + Optimized Threshold
retention_key = pd.read_csv('Customer_Retention_Key.csv') #Customer Retention Key

final_prob = best_xgb.predict_proba(X)[:, 1]

final_results = retention_key.copy()
final_results['Churn_Probability'] = final_prob
final_results['Predicted_Churn'] = (final_prob >= best_threshold).astype(int)

# High-risk customers
high_risk = final_results[final_results['Churn_Probability'] > best_threshold] \
    .sort_values(['Churn_Probability', 'Value_at_Risk'], ascending=False)

print(f"Number of high-risk customers (using threshold {best_threshold:.4f}): {len(high_risk)}")
display(high_risk.head(10))

# Save for the report
high_risk.to_csv('High_Risk_Customers_for_Retention.csv', index=False)

Number of high-risk customers (using threshold 0.5155): 2718


,customerID,Churn,Value_at_Risk,Churn_Probability,Predicted_Churn
4800,9300-AGZNL,1,94.00,0.975548,1
2208,7216-EWTRS,1,100.80,0.975424,1
3380,5178-LMXOP,1,95.10,0.974705,1
1976,9497-QCMMS,1,93.55,0.974160,1
3209,8149-RSOUN,1,93.85,0.971608,1
2577,4910-GMJOT,1,94.60,0.970522,1
6089,8775-LHDJH,1,90.60,0.969722,1
585,5192-EBGOV,1,85.70,0.969188,1
6866,0295-PPHDO,1,95.45,0.968259,1
6482,5419-JPRRN,1,101.45,0.968208,1
